# GeoPrompt Native Workflow

This notebook stays fully inside GeoPrompt so the default teaching path is independent and lightweight.

It also saves visible outputs that render directly in GitHub after the notebook is committed.

In [1]:
from pathlib import Path
import geoprompt as gp

output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

frame = gp.geopromptframe.from_records(
    [
        {'site_id': 'a', 'demand': 12.0, 'geometry': {'type': 'Point', 'coordinates': (-111.90, 40.75)}},
        {'site_id': 'b', 'demand': 9.0, 'geometry': {'type': 'Point', 'coordinates': (-111.84, 40.78)}},
        {'site_id': 'c', 'demand': 15.0, 'geometry': {'type': 'Point', 'coordinates': (-111.88, 40.73)}},
    ],
    crs='EPSG:4326',
)

print(frame.head())
print(frame.summary())
print('Nearest neighbors:', frame.nearest_neighbors(k=1))

[{'site_id': 'a', 'demand': 12.0, 'geometry': {'type': 'Point', 'coordinates': (-111.9, 40.75)}}, {'site_id': 'b', 'demand': 9.0, 'geometry': {'type': 'Point', 'coordinates': (-111.84, 40.78)}}, {'site_id': 'c', 'demand': 15.0, 'geometry': {'type': 'Point', 'coordinates': (-111.88, 40.73)}}]
{'row_count': 3, 'column_count': 3, 'columns': ['site_id', 'demand', 'geometry'], 'crs': 'EPSG:4326', 'geometry_types': ['Point'], 'bounds': {'min_x': -111.9, 'min_y': 40.73, 'max_x': -111.84, 'max_y': 40.78}, 'column_stats': [{'column': 'site_id', 'dtype': 'string', 'null_count': 0, 'unique_count': 3}, {'column': 'demand', 'dtype': 'numeric', 'null_count': 0, 'unique_count': 3, 'min': 9.0, 'max': 15.0, 'mean': 12.0}]}
Nearest neighbors: [{'origin': 'a', 'neighbor': 'c', 'distance': 0.028284271247471345, 'origin_geometry_type': 'Point', 'neighbor_geometry_type': 'Point', 'rank': 1, 'distance_method': 'euclidean'}, {'origin': 'b', 'neighbor': 'c', 'distance': 0.06403124237432685, 'origin_geometry_ty

In [2]:
report = gp.build_scenario_report(
    baseline_metrics={'served_load': 180.0, 'deficit': 0.12},
    candidate_metrics={'served_load': 205.0, 'deficit': 0.05},
    higher_is_better=['served_load'],
)
written = gp.export_scenario_report(report, output_dir / 'notebook-native-report.html')
print(gp.scenario_report_table(report).to_markdown())
print('Wrote:', written)

| metric | baseline | candidate | delta | delta_percent | direction |
| --- | --- | --- | --- | --- | --- |
| deficit | 0.12 | 0.05 | -0.06999999999999999 | -58.33333333333333 | improved |
| served_load | 180.0 | 205.0 | 25.0 | 13.88888888888889 | improved |

Wrote: D:\Github\geoprompt\outputs\notebook-native-report.html


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

rows = frame.to_records() if hasattr(frame, 'to_records') else list(frame)
labels = [row['site_id'] for row in rows]
values = [row['demand'] for row in rows]

plt.rcParams.update({
    'figure.facecolor': '#f8fafc',
    'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#e2e8f0',
    'axes.grid': True,
    'grid.color': '#f1f5f9',
    'grid.linewidth': 0.8,
    'font.family': 'sans-serif',
    'font.size': 10,
})

colors = ['#3b82f6', '#22c55e', '#7c3aed']
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors, edgecolor=['#2563eb', '#16a34a', '#6d28d9'],
              linewidth=0.5, width=0.55, zorder=3)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.0f}',
            ha='center', va='bottom', fontsize=11, color='#334155', fontweight='600')

ax.set_ylabel('Demand', fontsize=10, color='#334155')
ax.set_title('Native GeoPrompt — Demand by Site', fontsize=14, fontweight='bold', color='#0f172a', pad=14)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors='#64748b')
ax.set_ylim(0, max(values) * 1.2)
fig.tight_layout()

image_path = output_dir / 'notebook-native-summary.png'
fig.savefig(image_path, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
display(Image(filename=str(image_path)))
print('Saved:', image_path)

kernel-smoke-ok
